In [1]:
import sys
sys.path.append("../")

import torch
from orchard.config import load_config
from orchard.env import create_env
from orchard.model import create_networks
from orchard.seed import set_all_seeds
import orchard.encoding as encoding
from orchard.policy import argmax_a_Q_team, argmax_a_Q_team_batched

cfg = load_config(
    "/u/taddmao/code/orchard-action-market/systematic_debug_report/slurm_9x9/find_model_lambda_lr_that_works/config/base_dec.yaml",
    ["model.conv_specs=[[64,3],[64,3]]", "model.mlp_dims=[]",
     "logging.output_dir=/tmp/test"]
)

set_all_seeds(cfg.train.seed)
env = create_env(cfg.env)
encoding.init_encoder(cfg.model.input_type, cfg.env, cfg.model.k_nearest)

networks = create_networks(
    cfg.model, cfg.env, cfg.train.lr, cfg.train.total_steps,
    nstep=cfg.train.nstep, td_lambda=cfg.train.td_lambda,
    train_method=cfg.train.train_method, n_networks=cfg.env.n_agents,
)

# Test on 1000 random states
from orchard.enums import NUM_ACTIONS, Action
from orchard.seed import rng

mismatches = 0
s = env.init_state()
for i in range(1000):
    a_old = argmax_a_Q_team(s, networks, env)
    a_new = argmax_a_Q_team_batched(s, networks, env)
    if a_old != a_new:
        mismatches += 1
        print(f"MISMATCH at step {i}: old={a_old} new={a_new}")
    # advance to a new state
    action = Action(rng.randint(0, NUM_ACTIONS - 1))
    s = env.step(s, action).s_t_next

print(f"\nDone: {mismatches}/1000 mismatches")

MISMATCH at step 19: old=Action.UP new=Action.STAY
MISMATCH at step 94: old=Action.UP new=Action.STAY
MISMATCH at step 226: old=Action.LEFT new=Action.STAY
MISMATCH at step 234: old=Action.LEFT new=Action.STAY
MISMATCH at step 243: old=Action.RIGHT new=Action.STAY
MISMATCH at step 290: old=Action.LEFT new=Action.STAY
MISMATCH at step 298: old=Action.LEFT new=Action.STAY
MISMATCH at step 464: old=Action.DOWN new=Action.STAY
MISMATCH at step 465: old=Action.DOWN new=Action.STAY
MISMATCH at step 500: old=Action.DOWN new=Action.STAY
MISMATCH at step 501: old=Action.RIGHT new=Action.STAY
MISMATCH at step 516: old=Action.DOWN new=Action.STAY
MISMATCH at step 520: old=Action.DOWN new=Action.STAY
MISMATCH at step 524: old=Action.DOWN new=Action.STAY
MISMATCH at step 528: old=Action.DOWN new=Action.STAY
MISMATCH at step 533: old=Action.DOWN new=Action.STAY
MISMATCH at step 537: old=Action.DOWN new=Action.STAY
MISMATCH at step 541: old=Action.DOWN new=Action.STAY
MISMATCH at step 581: old=Action

In [2]:
cfg_cen = load_config(
    "/u/taddmao/code/orchard-action-market/systematic_debug_report/slurm_9x9/find_model_lambda_lr_that_works/config/base_cen.yaml",
    ["model.conv_specs=[[64,3],[64,3]]", "model.mlp_dims=[]",
     "logging.output_dir=/tmp/test_cen"]
)

set_all_seeds(cfg_cen.train.seed)
env_cen = create_env(cfg_cen.env)
encoding.init_encoder(cfg_cen.model.input_type, cfg_cen.env, cfg_cen.model.k_nearest)

networks_cen = create_networks(
    cfg_cen.model, cfg_cen.env, cfg_cen.train.lr, cfg_cen.train.total_steps,
    nstep=cfg_cen.train.nstep, td_lambda=cfg_cen.train.td_lambda,
    train_method=cfg_cen.train.train_method, n_networks=1,
)

mismatches = 0
s = env_cen.init_state()
for i in range(1000):
    a_old = argmax_a_Q_team(s, networks_cen, env_cen)
    a_new = argmax_a_Q_team_batched(s, networks_cen, env_cen)
    if a_old != a_new:
        mismatches += 1
        print(f"CEN MISMATCH at step {i}: old={a_old} new={a_new}")
    action = Action(rng.randint(0, NUM_ACTIONS - 1))
    s = env_cen.step(s, action).s_t_next

print(f"\nCentralized: {mismatches}/1000 mismatches")

CEN MISMATCH at step 6: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 21: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 56: old=Action.UP new=Action.STAY
CEN MISMATCH at step 57: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 69: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 133: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 177: old=Action.UP new=Action.STAY
CEN MISMATCH at step 181: old=Action.UP new=Action.STAY
CEN MISMATCH at step 188: old=Action.UP new=Action.STAY
CEN MISMATCH at step 237: old=Action.UP new=Action.STAY
CEN MISMATCH at step 244: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 256: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 274: old=Action.DOWN new=Action.STAY
CEN MISMATCH at step 276: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 294: old=Action.DOWN new=Action.STAY
CEN MISMATCH at step 298: old=Action.DOWN new=Action.STAY
CEN MISMATCH at step 320: old=Action.RIGHT new=Action.STAY
CEN MISMATCH at step 

In [3]:
# Pick a state that mismatches and compare Q-values
set_all_seeds(cfg.train.seed)
env2 = create_env(cfg.env)
encoding.init_encoder(cfg.model.input_type, cfg.env, cfg.model.k_nearest)

s = env2.init_state()
for i in range(44):  # get to step 43 (first mismatch)
    action = Action(rng.randint(0, NUM_ACTIONS - 1))
    s = env2.step(s, action).s_t_next

from orchard.enums import ACTION_PRIORITY
from orchard.policy import Q_team

print("=== Per-action Q-values (old path) ===")
for a in ACTION_PRIORITY:
    print(f"  {a.name:6s}: {Q_team(s, a, networks, env2):.8f}")

print("\n=== Per-action batched values ===")
after_states = [env2.apply_action(s, a) for a in ACTION_PRIORITY]
with torch.no_grad():
    for i_net, net in enumerate(networks):
        batch_enc = encoding.encode_batch_for_actions(s, i_net, after_states)
        vals = net(batch_enc)
        print(f"  net {i_net}: {vals.tolist()}")
        
        # Compare to individual encodes
        for k, a in enumerate(ACTION_PRIORITY):
            single_enc = encoding.encode(after_states[k], i_net)
            single_val = net(single_enc).item()
            batch_val = vals[k].item()
            if abs(single_val - batch_val) > 1e-6:
                print(f"    VALUE DIFF action={a.name}: single={single_val:.8f} batch={batch_val:.8f}")
                
                # Show grid diffs
                diff = (batch_enc.grid[k] - single_enc.grid).abs()
                if diff.sum() > 0:
                    nonzero = torch.nonzero(diff)
                    print(f"    Grid diffs at: {nonzero.tolist()}")

=== Per-action Q-values (old path) ===
  LEFT  : 0.06527641
  DOWN  : 0.06100998
  RIGHT : 0.04897065
  UP    : 0.05076027
  STAY  : 0.04897065

=== Per-action batched values ===
  net 0: [-0.02270631492137909, -0.015401558019220829, -0.024483107030391693, -0.025053925812244415, -0.02448311075568199]
  net 1: [0.047542281448841095, 0.04080341011285782, 0.04176747798919678, 0.04168516397476196, 0.04176749289035797]
  net 2: [0.040238864719867706, 0.033757977187633514, 0.03486190736293793, 0.03262139856815338, 0.03486189991235733]
  net 3: [0.0002016308717429638, 0.0018501984886825085, -0.0031756148673594, 0.0015076999552547932, -0.003175628837198019]


In [4]:
set_all_seeds(cfg.train.seed)
env3 = create_env(cfg.env)
encoding.init_encoder(cfg.model.input_type, cfg.env, cfg.model.k_nearest)

s = env3.init_state()
for i in range(1000):
    a_old = argmax_a_Q_team(s, networks, env3)
    a_new = argmax_a_Q_team_batched(s, networks, env3)
    if a_old != a_new:
        print(f"MISMATCH at step {i}: old={a_old} new={a_new}")
        print(f"Actor={s.actor}, Agent positions={s.agent_positions}")
        
        after_states = [env3.apply_action(s, a) for a in ACTION_PRIORITY]
        with torch.no_grad():
            for i_net in range(len(networks)):
                batch_enc = encoding.encode_batch_for_actions(s, i_net, after_states)
                for k, a in enumerate(ACTION_PRIORITY):
                    single_enc = encoding.encode(after_states[k], i_net)
                    diff = (batch_enc.grid[k] - single_enc.grid).abs().sum().item()
                    sdiff = 0.0
                    if batch_enc.scalar is not None and single_enc.scalar is not None:
                        sdiff = (batch_enc.scalar[k] - single_enc.scalar).abs().sum().item()
                    if diff > 1e-6 or sdiff > 1e-6:
                        print(f"  net{i_net} action={a.name}: grid_diff={diff:.6f} scalar_diff={sdiff:.6f}")
                        nonzero = torch.nonzero((batch_enc.grid[k] - single_enc.grid).abs() > 1e-6)
                        print(f"    Diff at [ch, row, col]: {nonzero.tolist()}")
        break
    
    action = Action(rng.randint(0, NUM_ACTIONS - 1))
    s = env3.step(s, action).s_t_next

MISMATCH at step 1: old=Action.UP new=Action.STAY
Actor=1, Agent positions=(Grid(row=6, col=3), Grid(row=0, col=7), Grid(row=3, col=2), Grid(row=5, col=2))


In [5]:
set_all_seeds(cfg.train.seed)
env3 = create_env(cfg.env)
encoding.init_encoder(cfg.model.input_type, cfg.env, cfg.model.k_nearest)

s = env3.init_state()
for i in range(1000):
    a_old = argmax_a_Q_team(s, networks, env3)
    a_new = argmax_a_Q_team_batched(s, networks, env3)
    if a_old != a_new:
        print(f"MISMATCH at step {i}: old={a_old} new={a_new}")

        # Old path Q-values
        after_states = [env3.apply_action(s, a) for a in ACTION_PRIORITY]
        old_qs = []
        with torch.no_grad():
            for k, a in enumerate(ACTION_PRIORITY):
                q = sum(net(encoding.encode(after_states[k], j)).item() 
                        for j, net in enumerate(networks))
                old_qs.append(q)

        # Batched path Q-values
        batch_qs = [0.0] * 5
        with torch.no_grad():
            for j, net in enumerate(networks):
                batch_enc = encoding.encode_batch_for_actions(s, j, after_states)
                vals = net(batch_enc)
                for k in range(5):
                    batch_qs[k] += vals[k].item()

        print(f"  {'Action':<8} {'Old Q':>14} {'Batch Q':>14} {'Diff':>14}")
        for k, a in enumerate(ACTION_PRIORITY):
            d = batch_qs[k] - old_qs[k]
            marker = " <-- TIE?" if abs(old_qs[k] - old_qs[ACTION_PRIORITY.index(a_old)]) < 1e-5 else ""
            print(f"  {a.name:<8} {old_qs[k]:>14.10f} {batch_qs[k]:>14.10f} {d:>14.2e}{marker}")
        print()
        if i > 300:
            break

    action = Action(rng.randint(0, NUM_ACTIONS - 1))
    s = env3.step(s, action).s_t_next

MISMATCH at step 35: old=Action.DOWN new=Action.STAY
  Action            Old Q        Batch Q           Diff
  LEFT       0.0032188650   0.0032189018       3.68e-08
  DOWN       0.0200360604   0.0200360455      -1.49e-08 <-- TIE?
  RIGHT      0.0093488866   0.0093489154       2.89e-08
  UP         0.0013130857   0.0013131159       3.03e-08
  STAY       0.0200360604   0.0200360706       1.02e-08 <-- TIE?

MISMATCH at step 47: old=Action.LEFT new=Action.STAY
  Action            Old Q        Batch Q           Diff
  LEFT       0.0057654469   0.0057654134      -3.35e-08 <-- TIE?
  DOWN       0.0057654469   0.0057654134      -3.35e-08 <-- TIE?
  RIGHT      0.0052639912   0.0052639367      -5.45e-08
  UP        -0.0015301420  -0.0015301635      -2.14e-08
  STAY       0.0057654469   0.0057654469       0.00e+00 <-- TIE?

MISMATCH at step 68: old=Action.DOWN new=Action.STAY
  Action            Old Q        Batch Q           Diff
  LEFT      -0.0031871432  -0.0031871363       6.98e-09
  DOWN    